In [ ]:
# import libraries
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_mongodb import MongoDBAtlasVectorSearch

from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langgraph.graph import MessagesState, StateGraph, END
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver


In [ ]:
# retrieve api key and initialize chat model
load_dotenv()

api_key = os.environ.get("")

llm = init_chat_model("gpt-4o-mini", model_provider="openai", api_key=api_key)

In [ ]:
# retrieve embeddings from model

embeddings = OpenAIEmbeddings(model="text-embedding-3-large", api_key=api_key)


In [ ]:
# retrieve connection to MongoDB search index

vector_store = MongoDBAtlasVectorSearch(
    embedding = embeddings,
    collection="Composable_Search_Index",
    index_name="Composable_Search_Index",
    relevance_score_fn="cosine"
)

print(vector_store)

In [ ]:
# using the pypdfloader split pages and preview loaded information.

pdf_loader = PyPDFLoader('pdfs/atlassian product dev doc.pdf') #
print(pdf_loader)
pdf_pages = pdf_loader.load_and_split()
print(pdf_pages)

pdf_pages[:10]

In [ ]:
index = vector_store.add_documents(documents=pdf_pages)
print(index)

In [ ]:
# Tool to retrieve search results from vector store and return a serialized output.

@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """Retrieve information related to a query."""
    docs = vector_store.similarity_search(query, k=1)

    serialized = "\n\n".join(
        (f"source: { doc.metadata}\n" f"content: {doc.page_content}")
        for doc in docs
    )
    print(serialized)
    return serialized, docs


In [ ]:
# Generate an AIMessage

def query_or_respond(state: MessagesState):
    """Tool call for retrieval or respond."""
    llm_with_tools = llm.bind_tools([retrieve])
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Retrieval

tools = ToolNode([retrieve])

# Generate response

def generate(state: MessageState):
    """Generate answer."""
    # Generate tool messages
    tool_messages = []
    for message in reversed(state["messages"]):
        if message.type == "tool":
            recent_tool_messages.append(message)
        else
        break

    tool_messages = recent_tool_messages[::-1]
    
    # Format prompt

    content = "\n\n".join(doc.content for doc in tool_messages)

    message = (
        "Provide output in paragraphs with a list for responses with multiple values. "
        "Use the following retrieved content to answer "
        "the question. If you don't know the answer, say that you "
        "don't know. Use three sentences maximum, five paragraphs maximum and keep the "
        "answer concise."
        "\n\n"
        f"{docs_content}"
    )
    
    conversation_messages = [
        message
        for message in state["messages"]
        if message.type in ("human", "system")
        or message
    ]

     # Run functions
    
    response = llm.invoke(prompt)
    return {"message": [response]}
    

In [ ]:
# Add the tools to the graph builder with node linking and compile the graph. 

graph_builder.add_node(query_or_respond)
graph_builder.add_node(tools)
graph_builder.add_node(generate)

graph_builder.set_entry_point("")
graph_builder.add_conditional_edges(
    "query_or_respond",
    tools_condition,
    {END: END, "tools", "generate"}
)

graph_builder.add_edge("tools", "generate")
graph_builder.add_edge("generate", END)

graph = graph_builder.compile()


In [ ]:
# Display graph 

from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid()))

In [ ]:
# chat history with memory saver

memory = MemorySaver()
graph = graph_builder.compile(checkpoint=memory)

# Specify an ID

config = {"configurable": {threat_id: "Y0"}}

In [ ]:
# input message a with graph stream

input_message_0 = "What is the current temperature in London?"

for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message_0}]},
    stream_mode="values",
    config=config
):


In [ ]:
# chat history with memory saver

memory = MemorySaver()
graph = graph_builder.compile(checkpoint=memory)

# Specify an ID

config = {"configurable": {threat_id: "Y1"}}

In [ ]:
# input message a with graph stream

input_message_1 = "What are the daily temperatures this month?"

for step in graph.stream(
    {"messages": [{"role": "user", "content": input_message_1}]},
    stream_mode="values",
    config=config
):


In [ ]:
# chat history with memory saver

memory = MemorySaver()
graph = graph_builder.compile(checkpoint=memory)

# Specify an ID

config = {"configurable": {threat_id: "Y2"}}